# Kaggle model proxy with a Kaggle API token

A **normal** notebook (not a Benchmark Task): Kaggle gives it no model proxy, so the project's
`kaggle` provider makes its own proxy key from your Kaggle API token
(`KAGGLE_PROXY_USE_API_TOKEN = True`, see `src/kaggle_proxy_auth.py`).

Tests: **1** one prompt, **2** several prompts at the same time (parallel direct calls),
**3** a batch (one `task.evaluate` over many prompts), **4** a key renewal.

### The only thing to do: give the token (one of the two)
* **Kaggle Secret (recommended):** Add-ons > Secrets > Add Secret, label `KAGGLE_API_TOKEN`,
  value = your token from https://www.kaggle.com/settings/api, then tick it for this notebook.
* **Or** paste it between the quotes of `KAGGLE_API_TOKEN_DIRECT` in the next cell
  (it is then saved with the notebook, so clear it afterwards).

Then **Run All**. The token and the proxy key are never printed.

In [ ]:
# Settings
KAGGLE_API_TOKEN_DIRECT = ""   # optional: paste the token here instead of using the Kaggle Secret

# Models to test; the first one that answers in step 1 is used for steps 2-4.
MODELS = ["google/gemini-3-flash-preview",   # on Kaggle's model list for token keys
          "openai/gpt-oss-120b",             # on the list; needs a reasoning level other than "none"
          "openai/gpt-6-luna"]               # the run's model; expected to be refused with a token key
REASONING_BY_MODEL = {"openai/gpt-oss-120b": "low"}

PROMPT = "What is 17 * 23? Answer with the number only."
PROMPTS = [f"What is {a} + {b}? Answer with the number only." for a, b in ((2, 3), (10, 7), (41, 1), (6, 9), (12, 30), (8, 8))]

REPO_URL = "https://github.com/mostafabehroozi/analogical_math_rag.git"
REPO_BRANCH = "analogical-consistency"

In [ ]:
# Setup: project code, kaggle-benchmarks, and proof that this notebook has no proxy of its own
import os, subprocess, sys, shutil, time, importlib.util

print("Proxy set by Kaggle for this notebook:", "MODEL_PROXY_URL" in os.environ)
for name in ("MODEL_PROXY_URL", "MODEL_PROXY_API_KEY", "MODEL_PROXY_EXPIRY_TIME"):
    os.environ.pop(name, None)  # nothing below may use a notebook proxy

if importlib.util.find_spec("kaggle_benchmarks") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle-benchmarks"], check=True)
from importlib.metadata import version
print("kaggle-benchmarks", version("kaggle-benchmarks"), "| python", sys.version.split()[0])

CODE_DIR = "/tmp/analogical_math_rag"
shutil.rmtree(CODE_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, CODE_DIR], check=True)
print("code commit:", subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

import copy
from config import CONFIG
from src import kaggle_proxy_auth as kpa
from src.api_manager import KaggleBenchmarksAPIManager

config = copy.deepcopy(CONFIG)
config.update({
    "KAGGLE_PROXY_USE_API_TOKEN": True,                       # the switch
    "KAGGLE_API_TOKEN": KAGGLE_API_TOKEN_DIRECT.strip(),      # "" = Kaggle Secret / environment
    "KAGGLE_MODEL_NAME_FINAL_SOLVER": MODELS[0],
    "KAGGLE_REASONING_EFFORT_BY_MODEL": REASONING_BY_MODEL,
    "MAX_API_RETRIES": 1, "API_RETRY_DELAY_SECONDS": 2,
    "GLOBAL_CONSECUTIVE_ERROR_LIMIT": 0, "GLOBAL_PERIODIC_PAUSE_INTERVAL_MINUTES": 0,
    "API_BATCHING_ENABLED": False,
})
credential = kpa.resolve_kaggle_credential(config)
print("Kaggle API token found in:", credential.source)

In [ ]:
# Proxy key from the token, and the account's Model Proxy budget
manager = KaggleBenchmarksAPIManager(config["KAGGLE_MODEL_QUOTAS"], config)
for balance in kpa.fetch_proxy_quota(credential).get("quotaBalances", []):
    print(f"  {balance.get('refillPeriod')}: ${balance.get('quotaUsed', 0):.2f} of ${balance.get('totalQuotaAllowed')} used")

SUMMARY = {}

def show(reply):
    return repr(reply["text"].strip()[:60]) if reply["text"] else f"{reply['error_type']}: {(reply['error_message'] or '')[:200]}"

In [ ]:
# 1) One prompt to each model
WORKING = []
for model in MODELS:
    t0 = time.time()
    reply = manager.generate_content(PROMPT, model, temperature=0.0)
    print(f"{model}: {reply['status']} in {time.time() - t0:.1f}s -> {show(reply)}")
    if reply["status"] == "SUCCESS":
        WORKING.append(model)
SUMMARY["1 single prompt"] = f"{len(WORKING)}/{len(MODELS)} models answered: {WORKING}"
MODEL = WORKING[0] if WORKING else None
print("\nmodel for the next steps:", MODEL)

In [ ]:
# 2) Several prompts at the same time: parallel direct calls (no batching)
from concurrent.futures import ThreadPoolExecutor

if MODEL:
    t0 = time.time()
    with ThreadPoolExecutor(len(PROMPTS)) as pool:
        replies = list(pool.map(lambda prompt: manager.generate_content(prompt, MODEL, temperature=0.0), PROMPTS))
    ok = sum(reply["status"] == "SUCCESS" for reply in replies)
    for prompt, reply in zip(PROMPTS, replies):
        print(f"  {prompt[:16]!r}: {reply['status']} -> {show(reply)}")
    SUMMARY["2 parallel direct calls"] = f"{ok}/{len(PROMPTS)} in {time.time() - t0:.1f}s"
    print(SUMMARY["2 parallel direct calls"])

In [ ]:
# 3) A batch: the calls are collected into one kaggle_benchmarks task.evaluate run in parallel
if MODEL:
    batched = manager.for_config({**config, "API_BATCHING_ENABLED": {"kaggle": True},
                                  "API_BATCH_KAGGLE_N_JOBS": len(PROMPTS)})
    t0 = time.time()
    with ThreadPoolExecutor(len(PROMPTS)) as pool:
        replies = list(pool.map(lambda prompt: batched.generate_content(prompt, MODEL, temperature=0.0), PROMPTS))
    ok = sum(reply["status"] == "SUCCESS" for reply in replies)
    for prompt, reply in zip(PROMPTS, replies):
        print(f"  {prompt[:16]!r}: {reply['status']} -> {show(reply)}")
    SUMMARY["3 batch"] = f"{ok}/{len(PROMPTS)} in {time.time() - t0:.1f}s"
    print(SUMMARY["3 batch"])

In [ ]:
# 4) Key renewal, as when a key expires: a new key, clients rebuilt, calls go on
if MODEL:
    before = manager.proxy_credentials.generation
    manager.proxy_credentials.invalidate()
    reply = manager.generate_content(PROMPT, MODEL, temperature=0.0)
    SUMMARY["4 key renewal"] = (f"key {before} -> {manager.proxy_credentials.generation}, "
                                f"call after renewal: {reply['status']} {show(reply)}")

print("\n===== SUMMARY =====")
for step, result in SUMMARY.items():
    print(f"{step}: {result}")
print(manager.proxy_credentials.describe())